# Used Cars Data Prep
**Tasks:** Load → Clean → Encode → Split → Baseline MAE

## Task 1 — Load, Inspect, and Clean

In [ ]:
import pandas as pd
import numpy as np

# --- Load data ---
url = "https://raw.githubusercontent.com/dsrscientist/dataset1/master/used_cars.csv"
try:
    df = pd.read_csv(url)
    print("Loaded from URL")
except Exception:
    try:
        import kagglehub
        path = kagglehub.dataset_download("manishkr1754/cardekho-used-car-data")
        import os
        csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
        df = pd.read_csv(os.path.join(path, csv_files[0]))
        print("Loaded from kagglehub:", path)
    except Exception as e:
        raise RuntimeError(f"Could not load dataset: {e}")

In [ ]:
# 1.1 — Shape, info, missing %
print("Shape:", df.shape)
print()
df.info()
print()
print("Missing % per column:")
print((df.isnull().sum() * 100 / len(df)).round(2))

In [ ]:
# 1.2 — Drop rows where selling_price is missing
df = df.dropna(subset=["selling_price"])
print(f"After dropping missing selling_price: {df.shape}")

In [ ]:
# 1.3 — Strip unit strings and convert mileage, engine, max_power to numeric
def strip_and_convert(series):
    """Remove unit strings and coerce to numeric."""
    cleaned = series.astype(str).str.replace(r"[^\d.]", "", regex=True).str.strip()
    return pd.to_numeric(cleaned, errors="coerce")

for col in ["mileage", "engine", "max_power"]:
    df[col] = strip_and_convert(df[col])
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)
    print(f"{col}: median={median_val:.2f}, nulls remaining={df[col].isnull().sum()}")

In [ ]:
# 1.4 — Remove outlier/junk selling prices
before = len(df)
df = df[(df["selling_price"] != 999999999) & (df["selling_price"] >= 10000)]
print(f"Removed {before - len(df)} rows with invalid selling_price. Shape: {df.shape}")

In [ ]:
# 1.5 — Drop duplicate rows
before = len(df)
df = df.drop_duplicates()
print(f"Dropped {before - len(df)} duplicate rows.")

print("\n--- Final shape after all cleaning ---")
print(df.shape)

## Task 2 — Encode Categorical Features

In [ ]:
# 2.1 — Label encode transmission_type: Manual=0, Automatic=1
df["transmission_type"] = df["transmission_type"].map({"Manual": 0, "Automatic": 1})
print("transmission_type value counts:")
print(df["transmission_type"].value_counts())

In [ ]:
# 2.2 — One-hot encode fuel_type and seller_type
df = pd.get_dummies(df, columns=["fuel_type", "seller_type"], drop_first=True)
print("Shape after one-hot encoding:", df.shape)

In [ ]:
# 2.3 — Print final column list
print("Final columns:")
print(df.columns.tolist())

### 2.4 — Explanation: `drop_first=True` in One-Hot Encoding

**Why `drop_first=True`?**  
When one-hot encoding a categorical variable with *k* distinct categories, only *k−1* binary columns are needed to represent all categories — the dropped category is implicitly represented when all remaining dummy columns are 0. Keeping all *k* columns creates **perfect multicollinearity** (the dummy columns sum to 1 for every row), which is known as the **dummy variable trap**. This makes the feature matrix singular, causing issues for linear models and inflating model coefficients.

**What does a row of all zeros in the dummy columns represent?**  
A row where every dummy column for a categorical feature is 0 represents the **reference (dropped) category**. For example, if `fuel_type_Diesel` and `fuel_type_CNG` are the retained columns (and `Petrol` was dropped), then a row with both equal to 0 means that car runs on **Petrol** — the baseline category. All other categories are interpreted relative to it.

## Task 3 — Split and Compute Baseline MAE

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

# 3.1 — Define X and y; drop non-numeric columns from X
y = df["selling_price"]
X = df.drop(columns=["selling_price"])

# Drop any remaining non-numeric columns (e.g. car_name)
non_numeric = X.select_dtypes(exclude=["number"]).columns.tolist()
if non_numeric:
    print(f"Dropping non-numeric columns: {non_numeric}")
    X = X.drop(columns=non_numeric)

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
# 3.2 — Train / Test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print(f"Train size: {X_train.shape[0]}  |  Test size: {X_test.shape[0]}")

In [ ]:
# 3.3 & 3.4 — Baseline MAE (predict mean of y_train for every test row)
baseline_pred = np.full(shape=len(y_test), fill_value=y_train.mean())
baseline_mae = mean_absolute_error(y_test, baseline_pred)

print(f"Baseline MAE: ₹{round(baseline_mae)}")